# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "03_systemic_pipeline_v1"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-29"
UPDATED = "2026-03-29"
GIT_HASH = "..."

DESCRIPTION = """ 
 Configuration-driven pipeline for building systemic (cross-asset) features from panel data.

This notebook orchestrates panel construction, systemic feature aggregation, validation, and versioned export, serving as the foundation for regime detection and portfolio research workflows."""

# 2. Imports


In [ ]:
import pandas as pd
import plotly.express as px

from quant_research.data.registry.universe_registry import get_all_assets
from quant_research.panels.builders.panel_builder import PanelBuilder
from quant_research.features.loaders.asset_feature_loader import FeatureLoader
from quant_research.systemic.builders.systemic_builder import SystemicBuilder
from quant_research.data.processed.loaders.asset_processed_loader import AssetProcessedDataLoader #ver si se puede eliminar
from quant_research.config.paths import SYSTEMIC_PATH 

# 3. CONFIGURATION


In [ ]:
CONFIG = {

    # ============================================================
    # PANEL CONFIG
    # ============================================================
    "panel": {

        # ----------------------------------------
        # ASSET UNIVERSE
        # ----------------------------------------
        "assets": [a.symbol for a in get_all_assets()],
        # alternativa:
        # "assets": ["SPY", "QQQ", "TLT", "GLD", "BTC", "ETH"]

        # ----------------------------------------
        # PANEL ALIGNMENT
        # ----------------------------------------
        "alignment": "union",
        # options:
        # "union"        → mantiene todas las fechas (más NaNs)
        # "intersection" → solo fechas comunes (menos NaNs, más limpio)

        # ----------------------------------------
        # FEATURE SELECTION (asset-level)
        # ----------------------------------------
        "features": {

            "momentum": {
                "raw": "MOM_63",     # feature base
                "z": "MOM_63_Z",     # versión normalizada
            },

            "volatility": {
                "z": "VOL_63_Z",
            },

            # podés agregar más familias:
            # "carry": {...}
            # "liquidity": {...}
        },

        # ----------------------------------------
        # NaN HANDLING (CRÍTICO)
        # ----------------------------------------
        "nan_handling": {

            # --- Warmup period (inicio del dataset)
            "warmup": {
                "method": "cut",
                # options:
                # "cut"   → corta hasta primer dato válido común
                # "keep"  → mantiene NaNs iniciales

                "based_on": "used_features",
                # options:
                # "used_features" → solo features necesarias
                # "all"           → todo el panel
            },

            # --- Calendar alignment
            "calendar": {
                "method": "ffill",
                # options:
                # "ffill" → forward fill (más usado en finance)
                # "bfill" → backward fill
                # "drop"  → elimina filas con NaNs
                # "none"  → no hace nada
            },

            # --- Final cleaning
            "final": {
                "method": "dropna",
                # options:
                # "dropna" → elimina cualquier NaN restante
                # "keep"   → mantiene NaNs
            }
        },
    },

    # ============================================================
    # SYSTEMIC CONFIG
    # ============================================================
    "systemic": {

        "features": [

            # ====================================================
            # TREND / MOMENTUM (cross-asset)
            # ====================================================

            {
                "name": "mean_mom",
                "type": "mean",
                "features": ["MOM_63"],
                # options:
                # "features": ["MOM_5", "MOM_21", "MOM_63"]
            },

            # ====================================================
            # VOLATILITY LEVEL
            # ====================================================

            {
                "name": "mean_vol",
                "type": "mean",
                "features": ["VOL_63_Z"],
            },

            # ====================================================
            # DISPERSION
            # ====================================================

            {
                "name": "dispersion_ret",
                "type": "dispersion",
                "features": ["RET_1"],
                # method options (en aggregator):
                # "std" | "mad" | "range"
            },

            {
                "name": "dispersion_mom",
                "type": "dispersion",
                "features": ["MOM_63_Z"],
            },

            # ====================================================
            # BREADTH (market participation)
            # ====================================================

            {
                "name": "breadth_mom",
                "type": "breadth",
                "features": ["MOM_63_Z"],
                "threshold": 0.0,
                # interpretation:
                # % de assets con momentum positivo
            },

            # ====================================================
            # CORRELATION (systemic risk)
            # ====================================================

            {
                "name": "corr_ret",
                "type": "correlation",
                "features": ["RET_1"],
                "window": 20,
                # options:
                # window = 10, 20, 63
            },

            {
                "name": "corr_mom",
                "type": "correlation",
                "features": ["MOM_63_Z"],
                "window": 20,
            },

            # ====================================================
            # OPTIONAL: NORMALIZATION / TRANSFORMS
            # ====================================================

            {
                "name": "mean_mom_z",
                "type": "zscore",
                "input": "mean_mom",
                "window": 20,
                # options:
                # window = 20, 63, 126
            },

            # podés agregar:
            # smoothing / ema (más adelante en regimes)
        ]
    },

    # ============================================================
    # OUTPUT CONFIG
    # ============================================================
    "output": {

        "base_path": SYSTEMIC_PATH,

        "run_name": "baseline",
        # examples:
        # "baseline"
        # "mom_short_exp"
        # "corr_regime_test"

        # opcional futuro:
        # "versioning": "timestamp" | "manual"
    }
}

# 4. Data Loading

In [ ]:
feature_loader = FeatureLoader()
processed_loader = AssetProcessedDataLoader()

builder = PanelBuilder(feature_loader, processed_loader)
# symbols = ["BTC", "SPY", "QQQ", "XLE"]

panel = builder.build_panel(
    assets=CONFIG["panel"]["assets"],
    families=list(CONFIG["panel"]["features"].keys()),
    include_processed=["RET_1", "PRICE"],
    alignment=CONFIG["panel"]["alignment"]
                                                   # NaN policy is keep for default, NaNs treatment is performed in 5.1 by panel_preparer
)

## 4.1 Panel data audit

In [ ]:
import pandas as pd
import numpy as np

def audit_panel(panel: pd.DataFrame):
    
    assert isinstance(panel.columns, pd.MultiIndex), "Panel must have MultiIndex columns"

    features = panel.columns.get_level_values(0).unique()
    assets = panel.columns.get_level_values(1).unique()

    report = {}

    # ============================================================
    # SUMMARY
    # ============================================================

    report["summary"] = {
        "shape": panel.shape,
        "n_features": len(features),
        "n_assets": len(assets),
        "start": str(panel.index.min()),
        "end": str(panel.index.max()),
        "n_dates": panel.index.nunique(),
    }

    # ============================================================
    # NaNs (GLOBAL)
    # ============================================================

    total_nans = panel.isna().sum().sum()
    total_values = panel.size

    report["summary"]["nan_ratio"] = float(total_nans / total_values)

    # ============================================================
    # PER FEATURE
    # ============================================================

    per_feature = []

    for feat in features:
        df_feat = panel[feat]

        nan_ratio = df_feat.isna().sum().sum() / df_feat.size

        per_feature.append({
            "feature": feat,
            "nan_ratio": float(nan_ratio),
            "mean": float(df_feat.mean().mean()),
            "std": float(df_feat.std().mean()),
            "min": float(df_feat.min().min()),
            "max": float(df_feat.max().max()),
        })

    report["per_feature"] = pd.DataFrame(per_feature)

    # ============================================================
    # PER ASSET
    # ============================================================

    per_asset = []

    for asset in assets:
        df_asset = panel.xs(asset, level=1, axis=1)

        nan_ratio = df_asset.isna().sum().sum() / df_asset.size

        per_asset.append({
            "asset": asset,
            "nan_ratio": float(nan_ratio),
            "mean": float(df_asset.mean().mean()),
            "std": float(df_asset.std().mean()),
        })

    report["per_asset"] = pd.DataFrame(per_asset)

    # ============================================================
    # COVERAGE CHECK
    # ============================================================

    coverage = panel.notna().T.groupby(level="feature").mean().T

    report["coverage"] = coverage

    # ============================================================
    # OUTLIER CHECK (simple z-score)
    # ============================================================

    z = (panel - panel.mean()) / panel.std()
    extreme = (np.abs(z) > 5).sum().sum()

    report["summary"]["extreme_values"] = int(extreme)

    return report

In [ ]:
audit = audit_panel(panel)
audit["summary"]

In [ ]:
audit["per_feature"].sort_values("nan_ratio", ascending=False)

In [ ]:
audit["per_asset"].sort_values("nan_ratio", ascending=False)

In [ ]:
audit["coverage"].tail()

# 5. Core Computations

## 5.1 NaNs Handling

In [ ]:
from quant_research.systemic.preprocessing.panel_preparer import PanelPreparer
from quant_research.systemic.builders.systemic_builder import SystemicBuilder

# --- Step 1: Prepare panel
preparer = PanelPreparer(CONFIG["panel"])
panel_prepared, nan_info = preparer.prepare(panel)



In [ ]:
audit = audit_panel(panel_prepared)
audit["summary"]

In [ ]:
print(type(panel_prepared.columns))

## 5.2 Compute Systemic Features

In [ ]:
# --- Step 2: Build systemic
builder = SystemicBuilder(CONFIG["systemic"]["features"])
df_systemic = builder.build(panel_prepared)

In [ ]:
for col in df_systemic.columns:
    print(col, df_systemic[col].notna().sum())

In [ ]:
# if config["nan_handling"]["final"]["method"] == "dropna":
df_systemic = df_systemic.dropna()

In [ ]:
df_systemic.head()

# 6. Systemic Dataset Audit

In [ ]:
import pandas as pd
import numpy as np

def audit_systemic(df: pd.DataFrame):
    
    report = {}

    # ============================================================
    # BASIC STRUCTURE
    # ============================================================

    assert isinstance(df, pd.DataFrame), "Systemic must be DataFrame"
    assert not df.empty, "Systemic DataFrame is empty"
    assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
    assert df.index.is_monotonic_increasing, "Index must be sorted"

    report["shape"] = df.shape
    report["start"] = str(df.index.min())
    report["end"] = str(df.index.max())

    # ============================================================
    # NaNs (STRICT)
    # ============================================================

    total_nans = df.isna().sum().sum()
    assert total_nans == 0, f"Systemic contains NaNs: {total_nans}"

    report["nan_check"] = "passed"

    # ============================================================
    # DUPLICATES
    # ============================================================

    assert not df.index.duplicated().any(), "Duplicate timestamps found"
    report["duplicates"] = "none"

    # ============================================================
    # CONSTANT FEATURES
    # ============================================================

    constant_cols = [col for col in df.columns if df[col].std() == 0]

    assert len(constant_cols) == 0, f"Constant features found: {constant_cols}"

    report["constant_features"] = "none"

    # ============================================================
    # EXTREME VALUES
    # ============================================================

    z = (df - df.mean()) / df.std()
    extreme = (np.abs(z) > 8).sum().sum()

    report["extreme_values"] = int(extreme)

    # no assert → solo warning
    if extreme > 0:
        print(f"⚠️ Warning: {extreme} extreme values detected")

    # ============================================================
    # VALUE RANGES (sanity)
    # ============================================================

    ranges = {}

    for col in df.columns:
        ranges[col] = {
            "min": float(df[col].min()),
            "max": float(df[col].max())
        }

    report["ranges"] = ranges

    # ============================================================
    # FINAL STATUS
    # ============================================================

    report["status"] = "PASSED"

    return report

In [ ]:
audit_sys = audit_systemic(df_systemic)
audit_sys

# 7. Export

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import hashlib
import numpy as np
import copy

In [ ]:
# ------------------------------------------------
# OUTPUT CONFIG
# ------------------------------------------------

output_cfg = CONFIG["output"]

base_path = Path(output_cfg["base_path"])
run_name = output_cfg["run_name"]

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

export_dir = base_path / f"{run_name}_{timestamp}"
export_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ------------------------------------------------
# SYSTEMIC Z-SCORE (post-processing)
# ------------------------------------------------

df_systemic_z = (df_systemic - df_systemic.mean()) / df_systemic.std()

In [ ]:
# ------------------------------------------------
# DATASET HASH
# ------------------------------------------------

hash_val = hashlib.md5(
    pd.util.hash_pandas_object(df_systemic, index=True).values
).hexdigest()

In [ ]:
import copy

config_for_hash = copy.deepcopy(CONFIG)

config_hash = hashlib.md5(
    json.dumps(config_for_hash, sort_keys=True, default=str).encode()
).hexdigest()

In [ ]:
# ------------------------------------------------
# SAFE CONFIG COPY
# ------------------------------------------------

metadata_config = copy.deepcopy(CONFIG)

# enriquecer panel config
metadata_config["panel"]["nan_handling"]["resolved"] = nan_info

metadata_config["panel"]["alignment_detail"] = {
    "method": CONFIG["panel"]["alignment"],
    "reason": "preserve full calendar for systemic analysis"
}

In [ ]:
metadata = {

    # ------------------------------------------------
    # IDENTIDAD
    # ------------------------------------------------
    "name": run_name,
    "created_at": datetime.utcnow().isoformat(),
    
    "dataset_hash": hash_val,
    "config_hash": config_hash,

    # ------------------------------------------------
    # DATASET INFO
    # ------------------------------------------------
    "data": {
        "n_rows": int(df_systemic.shape[0]),
        "n_cols": int(df_systemic.shape[1]),
        "columns": list(df_systemic.columns),
        
    },

    "date_range": {
        "start": str(df_systemic.index.min()),
        "end": str(df_systemic.index.max())
    },

    # ------------------------------------------------
    # SYSTEMIC LAYER
    # ------------------------------------------------
    "systemic": {
        "features": list(df_systemic.columns),
        "n_features": int(df_systemic.shape[1]),
        "source_config_path": "config.systemic.features"
    },

    # ------------------------------------------------
    # CONFIG SNAPSHOT
    # ------------------------------------------------
    "config": metadata_config,

    # ------------------------------------------------
    # AUDIT
    # ------------------------------------------------
    "audit": {

        "panel": {
            "nan_ratio": audit["summary"]["nan_ratio"],
            "extreme_values": audit["summary"]["extreme_values"],
        },

        "systemic": {
            "status": audit_sys["status"],
            "extreme_values": audit_sys["extreme_values"],
            "shape": audit_sys["shape"]
        }
    }
}

In [ ]:
# ------------------------------------------------
# SAVE PARQUET
# ------------------------------------------------

df_systemic.to_parquet(export_dir / "systemic.parquet")
df_systemic_z.to_parquet(export_dir / "systemic_z.parquet")

In [ ]:
def json_serializer(obj):
    import numpy as np
    import pandas as pd
    from pathlib import Path

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()

    if isinstance(obj, (pd.Timestamp,)):
        return obj.isoformat()

    if isinstance(obj, Path):
        return str(obj)

    # fallback
    return str(obj)
    
with open(export_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4, default=json_serializer)

## 7.1 Systemic Dataset export audit

In [ ]:
# ------------------------------------------------
# LOAD BACK
# ------------------------------------------------

df_loaded = pd.read_parquet(export_dir / "systemic.parquet")
df_loaded_z = pd.read_parquet(export_dir / "systemic_z.parquet")

# ------------------------------------------------
# VALIDATE
# ------------------------------------------------

assert np.allclose(df_loaded.values, df_systemic.values, equal_nan=True)
assert list(df_loaded.columns) == list(df_systemic.columns)
assert df_loaded.index.equals(df_systemic.index)

assert np.allclose(df_loaded_z.values, df_systemic_z.values, equal_nan=True)

In [ ]:
print("✅ Export completed")
print(f"📁 Path: {export_dir}")

print("\nShape:", df_systemic.shape)

print("\nNaN ratio:")
print(df_systemic.isna().mean())

print("\nPreview:")
print(df_systemic.head())

# 8. Visualization

In [ ]:
spy_price = panel["PRICE"]["SPY"]
spy_price = spy_price.loc[df_systemic.index]

In [ ]:
df_z = df_systemic_z

In [ ]:
spy_z = (spy_price - spy_price.mean()) / spy_price.std()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

features = df_z.columns
n = len(features)

fig = make_subplots(
    rows=n,
    cols=1,
    shared_xaxes=False,
    subplot_titles=[f"{col} vs SPY (Z)" for col in features]
)

for i, col in enumerate(features, start=1):
    
    # feature Z
    fig.add_trace(
        go.Scatter(
            x=df_z.index,
            y=df_z[col],
            name=col,
            line=dict(width=2)
        ),
        row=i, col=1
    )
    
    # SPY Z
    fig.add_trace(
        go.Scatter(
            x=spy_z.index,
            y=spy_z,
            name="SPY_Z",
            line=dict(dash="dot"),
            opacity=0.7
        ),
        row=i, col=1
    )

fig.update_layout(
    height=300 * n,
    title="Systemic Features (Z) vs SPY (Z)",
    showlegend=False,
    template = "plotly_dark"
)
fig.update_xaxes(
    dtick="M3",   # cada 3 meses
)

fig.show()